# CLIP: align images and captions

**Learning Transferable Visual Models From Natural Language Supervision** · Radford et al. · 2021

[Original paper](https://arxiv.org/abs/2103.00020)

**Learning goal:** Train two encoders with symmetric contrastive loss and test retrieval and shuffled pairs.

## Before starting

Softmax, cross entropy, and introductory VLM.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

CLIP aligns images and descriptions rather than generating captions token by token. Two encoders produce vectors in a shared space; matching pairs are favored relative to other batch pairs.

Other batch items supply negatives. If multiple captions describe the same image, a single-positive objective may introduce false negatives. We use unique synthetic pairs to study the objective without downloading pretrained CLIP.

## Reproduction boundary

CLIP-style objective on synthetic features; no image/text towers, web-scale data, or zero-shot benchmark.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`u = normalize(image_encoder(x))`, `v = normalize(text_encoder(t))`, and `S = exp(log_scale) * u @ v.T`.

`L = (CE(S, arange(B)) + CE(S.T, arange(B))) / 2`. Diagonal entries are positives; rows perform image→text and columns text→image retrieval. Normalization separates direction from magnitude. Temperature controls softmax sharpness but cannot replace good representations.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Generate two modalities from shared latent information. Linear encoders replace large vision/text towers.

In [ ]:
batch = 24
latent = torch.randn(batch, 6)
image_features = latent @ torch.randn(6, 12) + 0.01 * torch.randn(batch, 12)
text_features = latent @ torch.randn(6, 10) + 0.01 * torch.randn(batch, 10)
image_encoder = nn.Linear(12, 16); text_encoder = nn.Linear(10, 16)
log_scale = nn.Parameter(torch.tensor(2.0))
labels = torch.arange(batch)
def similarities():
    u = F.normalize(image_encoder(image_features), dim=-1)
    v = F.normalize(text_encoder(text_features), dim=-1)
    return log_scale.exp().clamp(max=100) * (u @ v.T)
scores = similarities()
print('similarity shape:', scores.shape)
assert scores.shape == (batch, batch)

## Step 2

Update both encoders. Retrieval accuracy here measures only the memorized batch.

In [ ]:
optimizer = torch.optim.AdamW([*image_encoder.parameters(), *text_encoder.parameters(), log_scale], lr=0.02)
history = []
for step in range(120):
    scores = similarities()
    loss = (F.cross_entropy(scores, labels) + F.cross_entropy(scores.T, labels)) / 2
    optimizer.zero_grad(set_to_none=True); loss.backward(); optimizer.step()
    history.append(loss.item())
with torch.no_grad():
    scores = similarities()
    accuracy = (scores.argmax(-1) == labels).float().mean().item()
print('training-batch retrieval:', accuracy)
assert history[-1] < history[0]
plt.imshow(scores.detach(), cmap='Reds'); plt.xlabel('text index'); plt.ylabel('image index'); plt.colorbar(); plt.show()

## Step 3

Permute captions while keeping the old labels: a controlled pairing diagnostic.

In [ ]:
permutation = torch.roll(torch.arange(batch), 1)
aligned_loss = F.cross_entropy(scores, labels)
wrong_pair_loss = F.cross_entropy(scores[:, permutation], labels)
print('aligned loss:', aligned_loss.item(), 'wrong-pair loss:', wrong_pair_loss.item())
assert wrong_pair_loss > aligned_loss
# Correct labels restore equivalence after a column permutation.
inverse = torch.argsort(permutation)
corrected = F.cross_entropy(scores[:, permutation], inverse)
assert torch.allclose(corrected, aligned_loss, atol=1e-6)

## Reading the result

A strong diagonal establishes alignment of the training pairs, not zero-shot performance on new images. Permutations and inverse label permutations help expose indexing bugs.

## Independent experiments

1. Remove normalization and track output norms.
2. Duplicate a pair: what happens to single-positive labels?
3. Generate held-out latent samples and evaluate retrieval.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

Without normalization, vector magnitude can change scores. Duplicates create ambiguous positives and need appropriate targets/loss. For testing, use new latent samples with the same modality generators; otherwise the task itself changes.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.